# Mistral-7B + RAG on Spider

Schema-aware Text-to-SQL experiment with reusable project classes.

## 1. Project setup

In [ ]:
from pathlib import Path

PROJECT_ROOT = next((p for root in [Path.cwd(), Path('/content/project'), Path('/content')] for p in [root, *root.glob('**/*')] if p.is_dir() and (p / 'src').is_dir() and (p / 'requirements.txt').is_file()), None)
assert PROJECT_ROOT is not None, 'Extract the project ZIP under /content first.'
%cd $PROJECT_ROOT
print('Project:', PROJECT_ROOT)

In [ ]:
!pip -q install -r requirements.txt
print('Restart the session once only if Colab reports that imported packages were replaced.')

## 2. Load prompts

In [ ]:
from pathlib import Path
from src import MistralExperiment, MistralKnowledgeRetriever, MistralRunner, MistralSQLEvaluator, evaluate_retriever

ROOT = Path.cwd()
prompts = MistralExperiment.load_prompts(ROOT / 'prompts/Mistral7b/spider_prompts.json')
assert len(prompts) == 108
print(len(prompts), 'questions |', sorted({row['db_id'] for row in prompts}))

## 3. Load 4-bit Mistral-7B

In [ ]:
runner = MistralRunner.load_4bit()
experiment = MistralExperiment(runner)
evaluator = MistralSQLEvaluator(ROOT / 'datasets')
print('Mistral loaded')

## 4. Baseline evaluation

In [ ]:
LIMIT = 3  # Use None after the smoke test
baseline_rows = experiment.run(prompts, limit=LIMIT)
baseline_eval, baseline_metrics = evaluator.evaluate(baseline_rows, dataset='spider')
baseline_metrics

## 5. Build the FAISS knowledge retriever

In [ ]:
retriever = MistralKnowledgeRetriever(
    ROOT / 'knowledge/spider_knowledge_base.txt', dataset='spider')
print('Knowledge chunks:', len(retriever.corpus), '| vectors:', retriever.index.ntotal)

## 6. RAG evaluation

In [ ]:
rag_rows = experiment.run(prompts, retriever=retriever, top_k=3, limit=LIMIT)
rag_eval, rag_metrics = evaluator.evaluate(rag_rows, dataset='spider')
rag_metrics

## 6A. Retriever evaluation (does not change RAG generation)

These component-level metrics use the gold SQL only to derive the tables required by each question. The existing FAISS retriever and `top_k=3` setting are unchanged. `Table Recall@K` measures required-table coverage, `Table Precision@K` measures how focused the retrieved table set is, and `MRR` measures how early the first chunk containing a required table appears.


In [ ]:
import json

retrieval_rows, retrieval_metrics = evaluate_retriever(
    prompts, retriever, top_k=3, limit=LIMIT
)
retrieval_metrics

retrieval_output = ROOT / 'results/mistral_spider_retrieval_metrics.json'
retrieval_output.write_text(json.dumps({
    'metrics': retrieval_metrics,
    'results': retrieval_rows,
}, indent=2, default=str))
print('Saved:', retrieval_output)


## 7. Save results

In [ ]:
baseline_output = ROOT / 'results/mistral_spider_baseline.json'
rag_output = ROOT / 'results/mistral_spider_rag.json'
evaluator.save(baseline_eval, baseline_metrics, baseline_output)
evaluator.save(rag_eval, rag_metrics, rag_output)
print('Saved:', baseline_output)
print('Saved:', rag_output)